In [35]:
!pip install -q torch transformers datasets accelerate

In [36]:
import torch
import json
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW

In [37]:
dataset = load_dataset("tatsu-lab/alpaca", split="train[:5000]")

In [38]:
def make_short(text):
    sentences = text.split(".")
    return sentences[0] + "."   # keep first 20 words

processed = []

for item in dataset:
    processed.append({
        "prompt": item["instruction"],
        "response": make_short(item["output"]),
        "original": item["output"]
    })

In [39]:
with open("train.json", "w") as f:
    json.dump(processed, f)

In [40]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [41]:
class MyDataset(Dataset):
    def __init__(self, file, tokenizer, max_len=128):
        with open(file) as f:
            self.data = json.load(f)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = f"Answer briefly and correctly in one sentence:\nQ: {item['prompt']}\nA: {item['response']}"
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels": enc["input_ids"].squeeze()
        }

In [42]:
train_dataset = MyDataset("train.json", tokenizer)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

In [43]:
optimizer = AdamW(model.parameters(), lr=5e-5)

In [44]:
model.train()

for epoch in range(3):
    print(f"\nEpoch {epoch+1}")

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss  

        loss.backward()
        optimizer.step()

        print("Loss:", loss.item())


Epoch 1
Loss: 7.742018222808838
Loss: 4.747982025146484
Loss: 2.865442991256714
Loss: 2.1019177436828613
Loss: 1.521536111831665
Loss: 1.0050543546676636
Loss: 1.3722102642059326
Loss: 1.4375934600830078
Loss: 1.238926887512207
Loss: 1.1617463827133179
Loss: 1.5708420276641846
Loss: 1.2518419027328491
Loss: 1.0243698358535767
Loss: 1.6699509620666504
Loss: 0.773304283618927
Loss: 1.0751409530639648
Loss: 1.0201412439346313
Loss: 1.0488457679748535
Loss: 0.9185695648193359
Loss: 0.8358763456344604
Loss: 0.9372654557228088
Loss: 1.0288739204406738
Loss: 0.7746750712394714
Loss: 1.6321908235549927
Loss: 0.8633373379707336
Loss: 1.6203504800796509
Loss: 0.9813998937606812
Loss: 0.6412274241447449
Loss: 1.036139965057373
Loss: 0.7535408735275269
Loss: 0.674584686756134
Loss: 0.7845656871795654
Loss: 0.8791953921318054
Loss: 0.6515687704086304
Loss: 0.7712259292602539
Loss: 0.7002379894256592
Loss: 0.8517648577690125
Loss: 1.1793783903121948
Loss: 0.7479651570320129
Loss: 1.305558681488037


In [45]:
def count_tokens(text):
    return len(tokenizer(text)["input_ids"])

sample = processed[0]

print("Original answer:\n", sample["original"])
print("\nShort answer:\n", sample["response"])

print("\nToken comparison:")
print("Original:", count_tokens(sample["original"]))
print("Short:", count_tokens(sample["response"]))

Original answer:
 1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. 
2. Exercise regularly to keep your body active and strong. 
3. Get enough sleep and maintain a consistent sleep schedule.

Short answer:
 1.

Token comparison:
Original: 45
Short: 2


In [63]:
model.eval()
prompt = "Answer briefly and correctly in one sentence:\nQ: Explain gravity\nA:"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

output = model.generate(
    **inputs,
    max_new_tokens=30,
    do_sample=False,
    no_repeat_ngram_size=2,
    eos_token_id=tokenizer.eos_token_id
)

print("\nGenerated Answer:\n")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Generated Answer:

Answer briefly and correctly in one sentence:
Q: Explain gravity
A: Gravity is a force which is distributed around the Earth.


In [64]:
# Your model output
pred = tokenizer.decode(output[0], skip_special_tokens=True)
pred = pred.split("A:")[-1].strip()

print("Model Answer:", pred)

Model Answer: Gravity is a force which is distributed around the Earth.


In [65]:
questions = [
    ("What is gravity?", "Gravity is a force that attracts objects with mass."),
    ("What is binary search?", "Binary search is an algorithm that repeatedly halves the search space."),
    ("What is AI?", "Artificial intelligence is the simulation of human intelligence by machines."),
    ("What is overfitting?", "Overfitting occurs when a model learns training data too well and fails on new data.")
]

In [66]:
def count_tokens(text):
    return len(tokenizer(text)["input_ids"])

def overlap_score(pred, ref):
    pred_words = set(pred.lower().split())
    ref_words = set(ref.lower().split())

    common = pred_words & ref_words

    if len(common) == 0:
        return 0

    precision = len(common) / len(pred_words)
    recall = len(common) / len(ref_words)

    return 2 * precision * recall / (precision + recall)

In [67]:
model.eval()

results = []

for q, ref in questions:
    prompt = f"Answer briefly and correctly in one sentence:\nQ: {q}\nA:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        no_repeat_ngram_size=2,
        eos_token_id=tokenizer.eos_token_id
    )

    pred = tokenizer.decode(output[0], skip_special_tokens=True)
    pred = pred.split("A:")[-1].strip()

    tokens = count_tokens(pred)
    score = overlap_score(pred, ref)

    print("\n----------------------")
    print("Q:", q)
    print("Model:", pred)
    print("Tokens:", tokens)
    print("Score:", score)

    results.append((tokens, score))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



----------------------
Q: What is gravity?
Model: Gravity is a force which is distributed around the Earth.
Tokens: 12
Score: 0.4444444444444444

----------------------
Q: What is binary search?
Model: Binary search is a type of data analysis where data is stored in a binary array.
Tokens: 18
Score: 0.2727272727272727


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



----------------------
Q: What is AI?
Model: AI is a field of computer science that aims to develop algorithms that enable machines to perform tasks that humans can't.
Tokens: 23
Score: 0.14814814814814814

----------------------
Q: What is overfitting?
Model: Overfitting is a type of software engineering technique used in software development.
Tokens: 14
Score: 0.15384615384615383


In [68]:
avg_tokens = sum(x[0] for x in results) / len(results)
avg_score = sum(x[1] for x in results) / len(results)

print("\n=== FINAL RESULT ===")
print("Average Tokens:", avg_tokens)
print("Average Score:", avg_score)


=== FINAL RESULT ===
Average Tokens: 16.75
Average Score: 0.25479150479150475
